In [ ]:
# conda activate anndata

import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import tqdm as notebook_tqdm

os.chdir("scRNA-seq")

In [ ]:
adata = ad.read_h5ad("data/matrix_scvi_annotated.h5ad") 

In [15]:
adata_anno = adata[adata.obs['cell_type'].str.contains("neuron")]
adata_anno.obs['cell_type'].unique()
adata_anno.var.head()

,gene_names,n_cells,highly_variable,highly_variable_rank,means,variances,variances_norm
ENSMUSG00000067879,3110035E14Rik,183,True,104.0,0.120930,0.435454,2.893598
ENSMUSG00000025912,Mybl1,236,True,1638.0,0.121353,0.161663,1.136421
ENSMUSG00000048960,Prex2,23,True,1671.0,0.010571,0.013002,1.133529
ENSMUSG00000067795,4930444P10Rik,7,True,1386.0,0.003383,0.004219,1.178026
ENSMUSG00000025939,Ube2w,1013,True,1409.0,0.889218,1.968264,1.175390


In [34]:
sc.tl.rank_genes_groups(
    adata_anno,
    groupby="cell_type",
    groups=["Excitatory neuron"],
    reference="Inhibitory neuron",
    method="wilcoxon"
)

In [35]:
# Extract result
result = adata_anno.uns['rank_genes_groups']
groups = result['names'].dtype.names

# Build mapping dict from .var
ens2sym = adata_anno.var['gene_names'].to_dict()

# Prepare DE results dict
de_results = {}

for group in groups:
    genes = result['names'][group]  # Ensembl IDs from DE
    logfc = result['logfoldchanges'][group]
    pvals = result['pvals_adj'][group]
    
    # Map to gene symbols, fallback to Ensembl ID if missing
    gene_symbols = [ens2sym.get(g, g) for g in genes]
    
    de_results[group] = {
        'gene_id': genes,
        'gene_symbol': gene_symbols,
        'logfoldchange': logfc,
        'pvals_adj': pvals
    }

In [ ]:
group = "Excitatory neuron"
df = pd.DataFrame(de_results[group])
df = df.sort_values(by='logfoldchange', ascending=False)
df.to_csv("data/DE_Excitatory_vs_Inhibitory.csv", index=False)

df.head()

,gene_id,gene_symbol,logfoldchange,pvals_adj
417,ENSMUSG00000037984,Neurod6,27.594330,0.000047
607,ENSMUSG00000038331,ENSMUSG00000038331,27.049721,0.001294
1776,ENSMUSG00000049555,Tmie,26.095793,0.306287
2355,ENSMUSG00000009214,Tmem8c,25.876926,0.593681
2079,ENSMUSG00000027568,ENSMUSG00000027568,25.847317,0.469654
